Load Data and Train-Test Split

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

X = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/train.csv', index_col='Id')
X_test = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/test.csv', index_col='Id')

X.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X.SalePrice
X.drop(['SalePrice'], axis=1, inplace=True)

X_train, X_valid, y_train, y_valid = train_test_split(X, y)

In [2]:
X_train.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
Id,,,,,,,,,,,,,,,,,,,,,
615,180,RM,21.0,1491,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
137,20,RL,NaN,10355,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,7,2007,WD,Normal
290,70,RL,60.0,8730,Pave,NaN,Reg,Lvl,AllPub,Inside,...,259,0,NaN,NaN,NaN,0,7,2007,WD,Normal
203,50,RL,50.0,7000,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,0,NaN,NaN,NaN,0,5,2006,COD,Normal
11,20,RL,70.0,11200,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal


In [3]:
y_train.head()

Id
615     75500
137    143000
290    153575
203    112000
11     129500
Name: SalePrice, dtype: int64

In [4]:
X_train.describe()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
count,1095.000000,902.000000,1095.000000,1095.000000,1095.000000,1095.000000,1095.000000,1088.000000,1095.000000,1095.000000,...,1095.000000,1095.000000,1095.000000,1095.000000,1095.000000,1095.000000,1095.000000,1095.000000,1095.000000,1095.000000
mean,58.041096,70.373614,10427.105936,6.076712,5.563470,1970.778995,1984.518721,96.399816,437.033790,45.606393,...,472.751598,90.565297,45.654795,22.884932,2.599087,15.658447,2.478539,39.053881,6.285845,2007.821005
std,42.766779,24.940575,10429.108356,1.376731,1.108113,30.312819,20.818915,169.485028,456.302583,155.404630,...,215.031753,121.044657,65.875482,63.272181,25.957020,56.231471,36.819477,325.982856,2.724312,1.331971
min,20.000000,21.000000,1300.000000,1.000000,1.000000,1880.000000,1950.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000
25%,20.000000,60.000000,7614.500000,5.000000,5.000000,1953.000000,1966.000000,0.000000,0.000000,0.000000,...,327.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,2007.000000
50%,50.000000,70.000000,9531.000000,6.000000,5.000000,1972.000000,1993.000000,0.000000,378.000000,0.000000,...,480.000000,0.000000,22.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000
75%,70.000000,80.000000,11406.500000,7.000000,6.000000,2000.000000,2004.000000,153.250000,707.000000,0.000000,...,576.000000,168.000000,66.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000
max,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1170.000000,5644.000000,1474.000000,...,1418.000000,670.000000,547.000000,552.000000,508.000000,480.000000,648.000000,8300.000000,12.000000,2010.000000


Seperate Numerical and Non-Numerical Data for Preprocessing

In [5]:
categorical_cols = [cname for cname in X_train.columns if
                    X_train[cname].nunique() < 10 and 
                    X_train[cname].dtype == "object"]

numerical_cols = [cname for cname in X_train.columns if 
                X_train[cname].dtype in ['int64', 'float64']]

my_cols = categorical_cols + numerical_cols
X_train = X_train[my_cols].copy()
X_valid = X_valid[my_cols].copy()
X_test = X_test[my_cols].copy()

Pipeline

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


numerical_transformer = SimpleImputer(strategy='constant')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
])

model = RandomForestRegressor(n_estimators=1000, random_state=0)

clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model', model)
                     ])

clf.fit(X_train, y_train)
preds = clf.predict(X_valid)

print('MAE:', mean_absolute_error(y_valid, preds))

MAE: 18863.941202739727
